# 19장 실습. 폐쇄망 RAG 라우터

외부 API 를 쓸 수 없는 망에서는 문서도 도메인별로 나눠 둔다. 진료 기록과 청구 자료와 규정 문서가 한 벡터 저장소에 섞여 있으면, 관계없는 문서가 검색에 끼어들고 접근 권한을 도메인별로 나누기도 어렵다. 라우터는 질문 하나를 읽고 어느 저장소를 뒤질지 먼저 정한다. 검색은 그 저장소 안에서만 하고, 찾은 구절은 6장의 재순위로 다시 줄 세운다.

```
질문 ─▶ 라우터 (target_domain: choice) ─▶ 그 도메인 저장소에서만 검색 (top-10) ─▶ 재순위 (6.2) ─▶ 답변 모델
           └─ 확신 < t ─▶ 상위 두 도메인을 함께 검색
```

| 절 | 내용 |
|---|---|
| 19.1 | 도메인별 저장소 — KLUE MRC 를 네 도메인으로 나눈다 |
| 19.2 | 라우터 판정 (Jev) |
| 19.3 | 격리 검색 — 라우팅한 저장소 대 통합 저장소 |
| 19.4 | 확신이 낮으면 두 도메인을 함께 |
| 19.5 | 재순위 |
| 19.6 | 용도별 증류 — KLUE MRC train 질문, Jev 교사와 출처 라벨을 섞는다 |
| 19.7 | 엔진 비교 — Jev · Laya 기본 · Laya 증류 · 로컬 LLM |

KLUE MRC 의 본문을 위키백과는 「백과」, 뉴스는 분류에 따라 「경제·금융」·「산업·기술」·「사회·국제」로 나눈다. 평가 질문은 dev 분할에서 도메인마다 50개씩 200개, 증류용 질문은 train 분할에서 뽑는다. 저장소는 TF-IDF 색인으로 대신한다. 임베딩 벡터 저장소로 바꿔도 라우터 부분은 그대로다.

검색 재료가 공개 데이터라 이 장의 실습은 Jev 를 부르지만, 실제 폐쇄망에서는 Jev 를 부를 수 없다. 이 장의 결론은 19.7 의 사내 엔진 세 줄에서 나온다.

In [1]:
import os, json, time, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, matplotlib.pyplot as plt, requests
import koreanize_matplotlib  # noqa: F401
from dotenv import load_dotenv
from requests.adapters import HTTPAdapter, Retry

load_dotenv()
DATA, OUT = Path("data"), Path("out")
OUT.mkdir(exist_ok=True)
TASKS = ["topic", "entails", "positive", "similarity"]
COLOR = {"topic": "#2a78d6", "entails": "#eb6834", "positive": "#1baf7a", "similarity": "#eda100"}
pd.set_option("display.precision", 3)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})
read_jsonl = lambda p: [json.loads(l) for l in Path(p).read_text(encoding="utf-8").splitlines() if l.strip()]

JEV_URL = os.getenv("JEV_URL", "https://api.typesafe.ai")
http = requests.Session()
http.mount(JEV_URL, HTTPAdapter(max_retries=Retry(total=4, backoff_factor=0.5, status_forcelist=[408, 429, 500, 502, 503, 504])))

def ask(state, questions, model="jev-latest"):
    r = http.post(f"{JEV_URL}/v1/systemone", timeout=60, headers={"Authorization": f"Bearer {os.environ['TYPESAFE_API_KEY']}"},
                  json={"model": model, "state": state, "questions": questions})
    r.raise_for_status()
    return r.json() | {"_latency_s": r.elapsed.total_seconds()}

choice = lambda instructions, criteria: {"type": "choice", "instructions": instructions, "criteria": criteria}
score = lambda instructions, levels: {"type": "score", "instructions": instructions, "criteria": levels}
noul = lambda instructions, true="", false="": {"type": "noul", "instructions": instructions} | ({"criteria": {"true": true, "false": false}} if true else {})

def probs_of(answer):
    return answer.get("probabilities") or {"true": answer["noul"], "false": 1 - answer["noul"]}

def judge(item, resp):
    (task, q), = item["questions"].items()
    a = resp["answers"][task]
    pr = {str(k): float(v) for k, v in probs_of(a).items()}
    best = max(pr, key=pr.get)
    pred = {"noul": lambda: pr["true"] >= 0.5, "choice": lambda: best, "score": lambda: int(best)}[q["type"]]()
    return {"id": item["id"], "task": item["task"], "qtype": q["type"], "gold": item["gold"], "model": resp["model"],
            "pred": pred, "probs": pr, "conf": pr[best], "conf_field": a.get("confidence"), "correct": bool(pred == item["gold"]),
            "input_tokens": resp["usage"]["input_tokens"], "latency_s": resp["_latency_s"]}

def run(items, engine=lambda it: ask(it["state"], it["questions"]), workers=4):
    with ThreadPoolExecutor(workers) as ex:
        return list(ex.map(lambda it: judge(it, engine(it)), items))

arrays = lambda recs: (np.array([r["conf"] for r in recs]), np.array([r["correct"] for r in recs], float))

def ece(conf, ok, bins=15):
    idx = np.clip(np.searchsorted(np.linspace(0, 1, bins + 1), conf, side="left") - 1, 0, bins - 1)
    return float(sum(abs(conf[idx == i].mean() - ok[idx == i].mean()) * (idx == i).sum() for i in np.unique(idx)) / len(conf))

import re, hashlib, torch
from laya.train import TrainConfig, finetune

MODELS = Path("models"); MODELS.mkdir(exist_ok=True)
(DATA / "distill").mkdir(exist_ok=True)
write_jsonl = lambda p, rows: Path(p).write_text("".join(json.dumps(r, ensure_ascii=False, default=str) + "\n" for r in rows), encoding="utf-8")
FULL = torch.cuda.is_available()
QKEYS = {"choice": lambda q: list(q["criteria"]), "score": lambda q: [str(i) for i in range(len(q["criteria"]))], "noul": lambda q: ["false", "true"]}
top = lambda a: max(probs_of(a), key=probs_of(a).get)          # 질문 유형과 관계없이 확률이 가장 큰 선택지
conf_of = lambda a: max(probs_of(a).values())
p_true = lambda a: probs_of(a)["true"]

def answer_all(engine, states, questions, workers=1):
    with ThreadPoolExecutor(workers) as ex:
        return list(ex.map(lambda s: engine(s, questions), states))

def teacher_rows(name, states, questions):
    # Jev 가 매긴 확률 분포를 그대로 교사 분포로 쓴다(13장 형식)
    resp = answer_all(ask, states, questions, workers=4)
    dist = lambda q, a: {"probabilities": {k: round(float(probs_of(a)[k]), 6) for k in QKEYS[q["type"]](q)}}
    rows = [{"id": f"{name}-{i:05d}", "state": s, "questions": questions, "gold": {n: dist(q, r["answers"][n]) for n, q in questions.items()}}
            for i, (s, r) in enumerate(zip(states, resp))]
    return rows, sum(r["usage"]["input_tokens"] for r in resp)

def distill(name, rows):
    # 학습 파일을 쓰고 multilingual 에서 파인튜닝한다. GPU 가 없으면 200행·1 epoch 의 빠른 모드로 돈다
    path = DATA / "distill" / f"{name}.jsonl"
    write_jsonl(path, random.Random(0).sample(rows, len(rows))[: [200, None][FULL]])
    t0 = time.perf_counter()
    rep = finetune(str(path), "multilingual", str(MODELS / name), TrainConfig(loss="rlcd", epochs=[1, 4][FULL], seed=0))
    return {"학습 항목(질문 단위)": rep["train_items"], "보정 항목": rep["calibration_items"], "epoch 손실": [round(x, 4) for x in rep["epoch_loss"]],
            "시간(초)": round(time.perf_counter() - t0, 1), "데이터 sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:16]}
print("장치:", torch.cuda.get_device_name(0) if FULL else "CPU", "| 모드:", ["빠른 모드", "전체"][FULL])

from sklearn.feature_extraction.text import TfidfVectorizer

RAW = DATA / "raw"
DOMAINS = {"백과": "인물·역사·지리·과학·문화 같은 백과사전식 지식",
           "경제·금융": "경기, 금융 시장, 부동산, 물가, 가계 경제에 관한 기사",
           "산업·기술": "기업, 산업, 자동차, IT·모바일, 게임, 신기술 제품에 관한 기사",
           "사회·국제": "사회 사건, 국제 뉴스, 문화·방송, 생활 소식에 관한 기사"}
NEWS_DOMAIN = {**dict.fromkeys(["경제", "금융", "생활경제", "부동산"], "경제·금융"),
               **dict.fromkeys(["산업섹션", "IT모바일", "테크", "게임", "자동차", "산업", "기업&CEO"], "산업·기술"),
               **dict.fromkeys(["사회", "국제", "문화/TV", "생활"], "사회·국제")}

장치: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition | 모드: 전체


## 19.1 도메인 격리 색인

본문을 두 문장씩 잘라 구절로 쓴다(6장과 같다). 질문의 정답은 (도메인, 그 도메인 안의 구절 번호)다. 분류가 「종합」·「기획」처럼 모호한 기사는 뺀다. 도메인마다 색인을 따로 만들고(격리), 비교를 위해 모든 구절을 한데 넣은 통합 색인도 만든다.

In [2]:
def two_sentences(text):
    sents = list(re.finditer(r"[^.!?。]+[.!?。]?\s*", text))
    return [(sents[i].start(), sents[min(i + 2, len(sents)) - 1].end()) for i in range(0, len(sents), 2)]

domain_of = lambda doc: "백과" if doc["source"] == "wikipedia" else NEWS_DOMAIN.get(doc.get("news_category"))

def load_mrc(name):
    docs = json.loads((RAW / name).read_text(encoding="utf-8"))["data"]
    chunks, qs = {d: [] for d in DOMAINS}, []
    for dom, doc in [(domain_of(doc), doc) for doc in docs if domain_of(doc)]:
        for para in doc["paragraphs"]:
            spans, base = two_sentences(para["context"]), len(chunks[dom])
            chunks[dom] += [para["context"][a:b_].strip() for a, b_ in spans]
            qs += [{"qid": qa["guid"], "question": qa["question"], "domain": dom, "gold": base + j}
                   for qa in para["qas"] if qa.get("answers") and not qa.get("is_impossible")
                   for j, (a, b_) in enumerate(spans) if a <= qa["answers"][0]["answer_start"] < b_]
    return chunks, qs

chunks, dev_qs = load_mrc("klue-mrc-v1.1_dev.json")
rng = random.Random(2026)
queries = [q for d in DOMAINS for q in rng.sample([q for q in dev_qs if q["domain"] == d], 50)]

def make_index(texts):
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=2, sublinear_tf=True)
    return v, v.fit_transform(texts)

search = lambda index, query, k=10: np.argsort(-(index[1] @ index[0].transform([query]).T).toarray().ravel())[:k]
stores = {d: make_index(texts) for d, texts in chunks.items()}
owner = [(d, i) for d, texts in chunks.items() for i in range(len(texts))]          # 통합 색인의 각 구절이 어느 도메인 몇 번인지
merged = make_index([t for texts in chunks.values() for t in texts])
pd.DataFrame({"구절 수": {d: len(v) for d, v in chunks.items()}, "평가 질문": pd.Series([q["domain"] for q in queries]).value_counts()})

,구절 수,평가 질문
백과,11154,50
경제·금융,6654,50
산업·기술,6545,50
사회·국제,7075,50


## 19.2 도메인 라우팅

질문 하나에 choice 하나를 묻는다. 선택지 설명에 도메인이 무엇을 담는지 적는다.

In [3]:
DQ = {"target_domain": choice("이 질문의 답이 들어 있을 문서 저장소", DOMAINS)}

def route_frame(rs):
    a = [r["answers"]["target_domain"] for r in rs]
    return pd.DataFrame({"질문": [q["question"] for q in queries], "정답": [q["domain"] for q in queries], "라우팅": [top(x) for x in a],
                         "확신": [conf_of(x) for x in a], "상위 둘": [sorted(probs_of(x), key=probs_of(x).get)[-2:][::-1] for x in a],
                         "지연(ms)": [1000 * r["_latency_s"] for r in rs]})

jev_route = answer_all(ask, [q["question"] for q in queries], DQ, workers=4)
R = route_frame(jev_route)
print("라우팅 정확도:", (R["라우팅"] == R["정답"]).mean(), "| 지연 중앙값(ms):", round(R["지연(ms)"].median(), 1))
pd.crosstab(R["정답"], R["라우팅"], margins=True)

라우팅 정확도: 0.635 | 지연 중앙값(ms): 160.8


라우팅,경제·금융,백과,사회·국제,산업·기술,All
정답,,,,,
경제·금융,27,4,11,8,50
백과,2,40,6,2,50
사회·국제,13,11,22,4,50
산업·기술,5,4,3,38,50
All,47,59,42,52,200


## 19.3 격리 검색 대 통합 검색

세 가지를 견준다. 라우팅한 저장소 하나에서 찾기, 정답 도메인을 미리 안다고 치고 찾기(라우터가 완벽할 때의 상한), 통합 저장소에서 찾기. 정답 구절이 상위 10개 안에 들면 성공이다(recall@10). 통합 저장소에서는 다른 도메인 구절이 결과에 얼마나 섞이는지도 본다.

In [4]:
hit = lambda domain, q, k=10: q["domain"] == domain and q["gold"] in search(stores[domain], q["question"], k)

merged_top = [[owner[i] for i in search(merged, q["question"])] for q in queries]
iso = pd.DataFrame({"라우팅 저장소": [hit(d, q) for d, q in zip(R["라우팅"], queries)],
                    "정답 도메인(상한)": [hit(q["domain"], q) for q in queries],
                    "통합 저장소": [(q["domain"], q["gold"]) in top_ for q, top_ in zip(queries, merged_top)]}).assign(정답=R["정답"])
display(iso.groupby("정답").mean().T.assign(전체=iso.drop(columns="정답").mean()))
print("통합 저장소 결과 중 다른 도메인 구절 비율:", np.mean([[d != q["domain"] for d, _ in top_] for q, top_ in zip(queries, merged_top)]))

정답,경제·금융,백과,사회·국제,산업·기술,전체
라우팅 저장소,0.36,0.62,0.30,0.42,0.425
정답 도메인(상한),0.66,0.82,0.64,0.54,0.665
통합 저장소,0.60,0.82,0.60,0.48,0.625


통합 저장소 결과 중 다른 도메인 구절 비율: 0.3825


## 19.4 저확신 다중 도메인 검색

라우터가 틀리면 검색이 통째로 빗나간다. 확신이 t 아래면 상위 두 도메인에서 5개씩 찾아 합친다. 뒤지는 저장소 수(평균)와 recall 을 t 마다 본다.

In [5]:
def routed_recall(R, t):
    doms = [[[r], two][c < t] for r, c, two in zip(R["라우팅"], R["확신"], R["상위 둘"])]
    return {"t": t, "recall@10": np.mean([any(hit(d, q, 10 // len(ds)) for d in ds) for q, ds in zip(queries, doms)]),
            "평균 저장소 수": np.mean([len(ds) for ds in doms])}

pd.DataFrame([routed_recall(R, t) for t in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 1.01]]).set_index("t")

,recall@10,평균 저장소 수
t,,
0.00,0.425,1.000
0.50,0.435,1.060
0.60,0.460,1.155
0.70,0.475,1.205
0.80,0.480,1.300
0.90,0.470,1.465
1.01,0.510,2.000


## 19.5 재순위화

라우팅한 저장소의 상위 10개를 6.2절의 재순위 질문으로 다시 줄 세운다. 질문 50개(네 개마다 하나)만 쓴다(구절 10개씩, 500번 판정).

In [6]:
RERANK_Q = {"answers": noul("`passage` 에 `question` 의 답이 들어 있다", true="passage 의 문장만으로 question 에 답할 수 있다",
                            false="passage 가 같은 주제를 다루더라도 question 의 답은 들어 있지 않다")}
sub = list(zip(queries, R["라우팅"]))[::4]
cand = [search(stores[d], q["question"]) for q, d in sub]
states = [{"question": q["question"], "passage": chunks[d][j]} for (q, d), c in zip(sub, cand) for j in c]
p = np.array([p_true(r["answers"]["answers"]) for r in answer_all(ask, states, RERANK_Q, workers=4)]).reshape(len(sub), -1)

# 정답 구절이 후보에 없으면(라우팅이 틀렸거나 검색이 놓쳤으면) 순위는 무한대다
pos = [list(c).index(q["gold"]) if q["domain"] == d and q["gold"] in c else -1 for (q, d), c in zip(sub, cand)]
before = np.array([np.inf if i < 0 else i + 1 for i in pos])
after = np.array([np.inf if i < 0 else (row > row[i]).sum() + 1 for i, row in zip(pos, p)])
pd.DataFrame({name: {"hit@1": (r == 1).mean(), "hit@3": (r <= 3).mean(), "MRR": np.mean(1 / r)} for name, r in [("검색 순서", before), ("재순위", after)]}).T

,hit@1,hit@3,MRR
검색 순서,0.20,0.34,0.290
재순위,0.42,0.48,0.447


## 19.6 용도별 증류

라우터의 학습 데이터는 만들 필요가 없다. KLUE MRC train 분할의 질문은 어느 기사·문서에서 나왔는지가 정해져 있고, 그 출처가 곧 도메인 정답이다. 다만 출처 라벨만으로 가르치면 「어느 저장소에 있어도 이상하지 않은」 질문(예: 기업 인물에 관한 질문)에도 확신하도록 배운다. 그래서 Jev 교사 분포와 출처 라벨(스무딩 0.1)을 반씩 섞는다(13장의 mix 와 같은 생각이다).

도메인마다 400개, 1,600개를 뽑는다. 평가 질문(dev)과는 분할이 달라 겹치지 않는다. 학습 전에 평가 질문 200개에 대한 로컬 LLM 의 답을 받아 두고 모델을 내린다.

In [7]:
_, train_qs = load_mrc("klue-mrc-v1.1_train.json")
train_q = [q for d in DOMAINS for q in random.Random(2026).sample([q for q in train_qs if q["domain"] == d], 400)]
pd.DataFrame([{"도메인": q["domain"], "질문": q["question"]} for q in train_q]).groupby("도메인").head(2)

,도메인,질문
0,백과,이명법'에 쓰이는 언어는 무엇인가?
1,백과,행정을 13도로 지정하고 황제의 권위를 높인 인물은?
400,경제·금융,김재원 의원의 친형이 현재 맡고있는 직책은?
401,경제·금융,포스트 코로나'가 테마인 행사가 시작되는 날짜는?
800,산업·기술,2분기 순익이 전년 대비 93%증가한 기업은?
801,산업·기술,프로모션이 진행되는 오프라인 채널은 몇 개인가?
1200,사회·국제,신흥국들의 급격한 통화가치 하락에 큰 영향을 미친 나라는?
1201,사회·국제,김영갑이 제주에 정착했을 때 나이는 몇 세였는가?


In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_MODEL = os.environ["LOCAL_LLM_MODEL"]
tok = AutoTokenizer.from_pretrained(LLM_MODEL, padding_side="left")
tok.pad_token = tok.pad_token or tok.eos_token
lm = AutoModelForCausalLM.from_pretrained(LLM_MODEL, dtype="auto", device_map="auto")

def chat_batch(prompts, max_new_tokens=256, batch_size=16, system=None):
    # 프롬프트 여럿을 묶어 generate 한 번에 보낸다(왼쪽 패딩). 서빙 엔진 없이 처리량을 내는 방법이다
    msgs = lambda p: [{"role": "system", "content": system}] * bool(system) + [{"role": "user", "content": p}]
    texts = [tok.apply_chat_template(msgs(p), tokenize=False, add_generation_prompt=True, enable_thinking=False) for p in prompts]
    outs = []
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i:i + batch_size], return_tensors="pt", padding=True).to(lm.device)
        gen = lm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in outs]

chat = lambda prompt, **kw: chat_batch([prompt], **kw)[0]
print(LLM_MODEL, "|", lm.device)

import re

KEYS = {"choice": lambda q: list(q["criteria"]), "score": lambda q: [str(i) for i in range(len(q["criteria"]))], "noul": lambda q: ["true", "false"]}
LINES = {"choice": lambda q: [f"  {k}: {v}" for k, v in q["criteria"].items()],
         "score": lambda q: [f"  {i}: {v}" for i, v in enumerate(q["criteria"])],
         "noul": lambda q: [f"  {k}: {v}" for k, v in (q.get("criteria") or {"true": "그렇다", "false": "아니다"}).items()]}

def gen_prompt(state, q):
    return "\n".join(["[판단할 대상]", json.dumps(state, ensure_ascii=False), "", "[질문]", q["instructions"], "[선택지]", *LINES[q["type"]](q), "",
                       "선택지 키 하나와, 그 답이 맞을 확률을 아래 두 줄 형식으로만 답하라.", "답: <선택지 키>", "확신: <0과 1 사이 숫자>"])

def gen_answer(q, text):
    keys = KEYS[q["type"]](q)
    ans = (re.findall(r"답\s*[:：]\s*(\S+)", text) + [None])[0]
    conf = float((re.findall(r"확신\s*[:：]\s*([01](?:\.\d+)?)", text) + ["nan"])[0])
    p = {k: [(1 - conf) / (len(keys) - 1), conf][k == ans] for k in keys} if ans in keys and conf == conf else {k: 1 / len(keys) for k in keys}
    return {"type": q["type"], "probabilities": p, "noul": p.get("true"), "choice": max(p, key=p.get), "invalid": ans not in keys, "raw": text}

def gen_many(states_questions, batch_size=16):
    # (state, 질문 하나짜리 dict) 목록을 묶어서 생성하고 Jev 모양의 응답으로 바꾼다
    flat = [(s, n, q) for s, qs in states_questions for n, q in qs.items()]
    t0 = time.perf_counter()
    outs = chat_batch([gen_prompt(s, q) for s, _, q in flat], max_new_tokens=24, batch_size=batch_size)
    sec = (time.perf_counter() - t0) / len(flat)
    return [{"model": "llm:" + LLM_MODEL, "answers": {n: gen_answer(q, o)}, "usage": {"input_tokens": 0}, "_latency_s": sec}
            for (s, n, q), o in zip(flat, outs)]

def llm_answers(states, questions, batch_size=16):
    flat, n = gen_many([(s, questions) for s in states], batch_size), len(questions)
    return [{"model": flat[0]["model"], "answers": {k: v for r in flat[i:i + n] for k, v in r["answers"].items()},
             "usage": {"input_tokens": 0}, "_latency_s": sum(r["_latency_s"] for r in flat[i:i + n])} for i in range(0, len(flat), n)]

def lines_of(text):
    # 생성한 목록에서 번호·기호를 떼고 한 줄에 하나씩 꺼낸다
    return [l for l in (re.sub(r"^\s*(\d+[.)]|[-*•])\s*", "", x).strip().strip('"') for x in text.splitlines()) if 4 <= len(l) <= 200]

generate = lambda prompts, max_new_tokens=512: chat_batch(prompts, max_new_tokens=max_new_tokens, batch_size=16)

llm_route = llm_answers([q["question"] for q in queries], DQ)
del lm; torch.cuda.empty_cache()

/home/josh/anaconda3/envs/pytorch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Ignoring corrupted tree cache file /data/shared/hf/hub/models--Qwen--Qwen3.5-4B/trees/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a.json: [Errno 13] Permission denied: '/data/shared/hf/hub/models--Qwen--Qwen3.5-4B/trees/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a.json'
Loading weights: 100%|██████████| 426/426 [00:01<00:00, 378.98it/s]


Qwen/Qwen3.5-4B | cuda:0


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


In [9]:
rows, tokens = teacher_rows("rag", [q["question"] for q in train_q], DQ)
hard = lambda dom: {d: 0.9 * (d == dom) + 0.1 / len(DOMAINS) for d in DOMAINS}
jev_top = [max(g := r["gold"]["target_domain"]["probabilities"], key=g.get) for r in rows]
for r, q in zip(rows, train_q):
    jp = r["gold"]["target_domain"]["probabilities"]
    r["gold"]["target_domain"]["probabilities"] = {d: round(0.5 * jp[d] + 0.5 * hard(q["domain"])[d], 6) for d in DOMAINS}
print(f"학습 행 {len(rows)} | Jev 판정과 출처 일치 {np.mean([j == q['domain'] for j, q in zip(jev_top, train_q)]):.3f} | Jev 비용 {tokens * 0.042 / 1e6:.4f}달러")
pd.Series(distill("rag-distill", rows))

학습 행 1600 | Jev 판정과 출처 일치 0.625 | Jev 비용 0.0302달러


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 2359.27it/s]


train items 1440, calibration items 160, eval items 160 (calibration_slice), skipped {}, device cuda:0
optimizer budget: 1440 train item(s), effective batch up to 64 (micro_batch 8 x grad_accum 8), 92 optimizer updates over 4 epoch(s)


/home/josh/anaconda3/envs/pytorch/lib/python3.11/site-packages/laya/train.py:724: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


epoch 1/4 step 100 loss 0.6204
epoch 1/4 mean loss 1.0214
epoch 2/4 step 100 loss 0.8395
epoch 2/4 mean loss 0.7552
epoch 3/4 step 100 loss 0.7318
epoch 3/4 mean loss 0.7051
epoch 4/4 step 100 loss 0.7123
epoch 4/4 mean loss 0.6943

=== Calibration Evidence (in-sample calibration slice, 160 items; not held-out) ===
Metric                 Before         After          Delta         
-----------------------------------------------------------------
Accuracy               0.5375         0.7000         +0.1625       
Loss                   1.4963         0.8900         -0.6063       
ECE                    0.2429         0.1276         -0.1153       
Brier (multi-class)    0.3856         0.1602         -0.2254       
Brier (top-1)          0.2860         0.1751         -0.1109       
Mean Confidence        0.7579         0.6540         -0.1039       
-----------------------------------------------------------------



학습 항목(질문 단위)                                1440
보정 항목                                        160
epoch 손실        [1.0214, 0.7552, 0.7051, 0.6943]
시간(초)                                       75.5
데이터 sha256                      b40e492757e331f2
dtype: object

## 19.7 엔진 비교

라우터 자리의 엔진만 바꾼다. 검색과 색인은 같다. 확신 0.7 아래에서 두 도메인을 함께 뒤지는 19.4 의 정책도 엔진마다 잰다.

In [10]:
import laya
from laya import Router

router = Router()                          # 한국어는 multilingual 체크포인트로 간다(10장)

def laya_post(state, questions):
    t0 = time.perf_counter()
    r = router.predict(state, questions)
    return r | {"model": "laya:" + r["routing"]["model"], "usage": {"input_tokens": 0}, "_latency_s": time.perf_counter() - t0}
print("laya", laya.__version__)

import laya

def checkpoint_engine(path, **load_kwargs):
    agent = laya.load(str(path), **load_kwargs)
    def post(state, questions):
        t0 = time.perf_counter()
        r = agent.predict(state, questions)
        return r | {"model": "laya:" + Path(path).name, "usage": {"input_tokens": 0}, "_latency_s": time.perf_counter() - t0}
    post.agent = agent
    return post

ENGINES = {"Jev": ask, "Laya 기본": laya_post, "Laya 증류": checkpoint_engine(MODELS / "rag-distill")}
resp = {n: answer_all(e, [q["question"] for q in queries], DQ, {"Jev": 4}.get(n, 1)) for n, e in ENGINES.items()} | {"로컬 LLM": llm_route}

def rag_report(rs):
    R_ = route_frame(rs)
    return {"라우팅 정확도": (R_["라우팅"] == R_["정답"]).mean(), "recall@10 (라우팅 저장소)": np.mean([hit(d, q) for d, q in zip(R_["라우팅"], queries)]),
            **{f"{k} (t=0.7)": v for k, v in routed_recall(R_, 0.7).items() if k != "t"}, "지연 중앙값(ms)": R_["지연(ms)"].median()}

compare = pd.DataFrame({n: rag_report(rs) for n, rs in resp.items()}).T
compare.loc["통합 저장소 (라우터 없음)", "recall@10 (라우팅 저장소)"] = iso["통합 저장소"].mean()
compare

laya 0.4.7


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 2215.93it/s]


,라우팅 정확도,recall@10 (라우팅 저장소),recall@10 (t=0.7),평균 저장소 수 (t=0.7),지연 중앙값(ms)
Jev,0.640,0.430,0.475,1.220,169.057
Laya 기본,0.455,0.280,0.350,1.420,13.162
Laya 증류,0.620,0.405,0.490,1.620,12.976
로컬 LLM,0.470,0.280,0.290,1.105,64.187
통합 저장소 (라우터 없음),NaN,0.625,NaN,NaN,NaN


## 정리

이 노트북이 잰 것과 만든 것이다. 결과 해석은 19장 본문에서 한다.

- 19.1 도메인별 구절 수와 색인
- 19.2 Jev 라우팅 정확도와 혼동표
- 19.3 라우팅 저장소·정답 도메인·통합 저장소의 recall@10, 통합 검색에 섞이는 다른 도메인 비율
- 19.4 확신 임계값별 두 도메인 검색의 recall 과 저장소 수
- 19.5 라우팅 저장소 결과의 재순위 전후 hit@1·MRR
- 19.6 train 질문 1,600개, Jev 와 출처의 일치율, 증류 체크포인트(`models/rag-distill`)
- 19.7 네 엔진의 라우팅 정확도·recall·지연